In [ ]:
import kagglehub
import numpy as np
import pandas as pd
import os
import time
import multiprocessing as mp
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, recall_score, roc_auc_score
from itertools import product

In [ ]:
# Descargar el conjunto de datos completo
dataset_root_path = kagglehub.dataset_download(
    "meowmeowmeowmeowmeow/gtsrb-german-traffic-sign"
)

## Reducción de datos
Vamos a reducir el dataset original de a 5k imagenes

In [ ]:
train_csv = os.path.join(dataset_root_path, "Train.csv")
df = pd.read_csv(train_csv)

# Pasaremos (por ahora) de 39k imagenes a 5k
df_muestra = df.sample(n=30000, random_state=42)

## Preprocesamiento

In [ ]:
from PIL import Image

X = []
y = []

# iteramos sobre el df para cargar, redimensionar y coleccionar las imagenes y etiquetas
for index, row in df_muestra.iterrows():
    # construimos la ruta completa hacia las imagenes
    image_path = os.path.join(dataset_root_path, row['Path'])

    try:
        # cargar la imagen
        img = Image.open(image_path)
        img = img.resize((96, 96)) # primero fue de 32px-> 64px, ->96px
        # Convertimos la imagen a un arreglo para integrarla a X
        X.append(np.array(img))
        # Integramos ClassId a y
        y.append(row['ClassId'])
    except Exception as e:
        print(f"Error procesando {image_path}: {e}")

# Convertir las listas a arreglos
X = np.array(X)
y = np.array(y)

In [ ]:
import cv2
# aplicaremos filtro gaussiano a cada imagen en X. Inicialmente se aplicó (5,5) pero no dio buenos resultados, reducimos a (3,3)
X_smoothed = np.array([cv2.GaussianBlur(img, (3, 3), 0) for img in X])

### SIFT

Es más comun realizar sift sobre grises perp considerando que el color es importante para este conjunto ya que colores como amarillo, rojos y blancos son de importancia visual va a dejar esta configuracion. Inicialmente sí se hizo sobre grises pero no dio buenos resultados

In [ ]:
import cv2

# inicializar sift
sift = cv2.SIFT_create()
all_descriptors = []

for i, img_gray in enumerate(X_smoothed):
    # Detectar keypoints y descriptores
    keypoints, descriptors = sift.detectAndCompute(img_gray, None)
    if descriptors is not None:
        all_descriptors.append(descriptors)

if all_descriptors:
    all_descriptors_np = np.vstack(all_descriptors)
else:
    all_descriptors_np = np.array([])

### Creación de vocabulario de bolsa de palabras visuales (BoVW)
Ahora, crearemos un vocabulario visual agrupando los descriptores SIFT mediante K-Means

In [ ]:
from sklearn.cluster import MiniBatchKMeans

# definimos el numero de palabras visuales (clusters)
k = 1000

if all_descriptors_np.shape[0] > 0:
    kmeans = MiniBatchKMeans(n_clusters=k, random_state=42, n_init='auto', verbose=False)
    kmeans.fit(all_descriptors_np)
    visual_vocabulary = kmeans.cluster_centers_
else:
    visual_vocabulary = None

### Generación de vectores de características BoVW
Tras crear el vocabulario, representaremos cada imagen como un histograma de estas palabras visuales.

In [ ]:
if visual_vocabulary is not None:
    bow_extractor = cv2.BOWImgDescriptorExtractor(sift, cv2.BFMatcher(cv2.NORM_L2))
    bow_extractor.setVocabulary(visual_vocabulary)

    bovw_features = []
    for img_gray in X_smoothed:
        keypoints = sift.detect(img_gray, None)
        if keypoints:
            features = bow_extractor.compute(img_gray, keypoints)
            if features is not None:
                bovw_features.append(features.flatten())
            else:
                bovw_features.append(np.zeros(k))
        else:
            bovw_features.append(np.zeros(k))

    bovw_features_np = np.array(bovw_features)
else:
    bovw_features_np = None

### Análisis de Componentes Principales (PCA)
Finalmente, aplicaremos PCA para reducir la dimensionalidad de los vectores de características BoVW.

In [ ]:
from sklearn.decomposition import PCA

if bovw_features_np is not None and bovw_features_np.shape[0] > 0:
    pca = PCA(n_components=0.95, random_state=42)
    bovw_features_pca = pca.fit_transform(bovw_features_np)
else:
    bovw_features_pca = None

### División de Datos y Línea Base Secuencial

In [ ]:
from sklearn.preprocessing import StandardScaler

if bovw_features_pca is not None:
    # División del dataset (80% entrenamiento, 20% prueba)
    X_train, X_test, y_train, y_test = train_test_split(
        bovw_features_pca, y, test_size=0.2, random_state=42
    )
    
    # Escalado de características (Necesario para SVM)
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    
    # --- Entrenamiento Secuencial (Línea Base para Métricas) ---
    start_seq = time.time()
    model_seq = SVC(kernel='rbf', C=10, gamma='scale', random_state=42)
    model_seq.fit(X_train, y_train)
    t_seq = time.time() - start_seq

### Entrenamiento en Paralelo y Evaluación de Métricas de Escalabilidad

In [ ]:
# Paralelo
def train_subset(data):
    X_sub, y_sub = data
    model = SVC(
    kernel='rbf',
    C=10,
    gamma='scale',
    random_state=42
    )
    model.fit(X_sub, y_sub)
    return model

tiempos = []
accura = []
speedups = []
eficiencias = []
karp_flatts = []

NUM_PROCESOS = 7

for i in range(2, NUM_PROCESOS + 1):
  # dividir dataset dinámicamente con base en 'i'
  indices = np.array_split(np.arange(len(X_train)), i)
  data_splits = [(X_train[idx], y_train[idx]) for idx in indices]

  start_par = time.time()

  with mp.Pool(processes=i) as pool:
      models = pool.map(train_subset, data_splits)

  t_par = time.time() - start_par
  tiempos.append(t_par)

  p = i 
  print(f" procesadores: {p}")
  
  #1 Speed-up
  sp = t_seq / t_par 
  speedups.append(sp)
  
  #2 Eficiencia
  ef = (sp / p) * 100
  eficiencias.append(ef)
  
  #3 Métrica Karp-Flatt
  kf = ((1 / sp) - (1 / p)) / (1 - (1 / p)) if sp != 1 else 0.0
  karp_flatts.append(kf)

  print(f"Tiempo Paralelo: {t_par:.4f} seg")
  print(f"Speed-up: {sp:.2f}")
  print(f"Eficiencia: {ef:.2f}%")
  print(f"Karp-Flatt: {kf:.4f}\n")

acc_ind = []
for _, model in enumerate(models):
    pred_par = model.predict(X_test)
    acc_par = accuracy_score(y_test, pred_par)
    acc_ind.append(acc_par)

acc_par = np.mean(acc_ind)
tiempo_prom = np.mean(tiempos)
accura.append(acc_par)